## 04. 메모리 기반 협업 필터링: 아티스트 기반
사용자의 청취 이력을 바탕으로 청취 패턴이 비슷한 아티스트를 추천한다. 기본 방식의 추천 결과와 성능을 확인한 뒤, 유사 아티스트 이웃 수에 따른 추가 실험을 진행한다.
- 입력 데이터: outputs/train.csv, outputs/test.csv
- 선호도: 전처리에서 계산한 preference = log(1 + plays)를 사용한다.
- 유사도: 아티스트별 사용자 선호도 벡터의 코사인 유사도를 계산한다.
- 추천 점수: 기존 청취 아티스트의 선호도와 후보 아티스트의 유사도를 곱한 뒤 합산한다.
- 기본 평가: 공통 evaluate.py를 사용해 인기순 추천과 Precision@10, Recall@10을 비교한다.

### 데이터 불러오기 및 확인

학습 데이터와 평가 데이터를 불러온 뒤 사용자–아티스트 쌍의 중복 여부와 분리 상태를 확인한다. 선호도는 이미 로그 변환된 값을 사용하며, 다시 로그 변환하지 않는다. 유사도와 추천 점수 계산에는 학습 데이터만 사용한다.

In [14]:
import copy
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from evaluate import evaluate
from memory_cf import ItemBasedCF

OUTPUT_DIR = Path("outputs")
K = 10
SETTINGS = {"all": None, "top10": 10, "top20": 20, "top50": 50}

train = pd.read_csv(OUTPUT_DIR / "train.csv")
test = pd.read_csv(OUTPUT_DIR / "test.csv")
print("train:", train.shape, "test:", test.shape)
assert not train.duplicated(["user_id", "artist_name"]).any()
assert not test.duplicated(["user_id", "artist_name"]).any()
assert train.merge(test, on=["user_id", "artist_name"]).empty, "train/test 쌍 중복"
assert set(test["user_id"]) <= set(train["user_id"])
assert set(test["artist_name"]) <= set(train["artist_name"]), "test에만 있는 아티스트 확인 필요"
np.testing.assert_allclose(train["preference"], np.log1p(train["plays"]))


train: (224925, 4) test: (56727, 4)


## 4.1 사용자–아티스트 행렬과 코사인 유사도
R[u, i]는 사용자 u의 아티스트 i에 대한 로그 선호도다. 아티스트 간 유사도는 행렬의 두 열을 비교하므로 cosine_similarity(R.T)로 계산한다.

비관측 칸의 0은 청취 정보가 없다는 의미이며, 명시적인 비선호로 해석하지 않는다. 다만 희소 데이터에서는 유사도 추정에 한계가 있다.

자기 자신과의 유사도는 0으로 설정한다. 기본 방식은 이웃 수를 제한하지 않고 모든 양의 유사도를 사용한다. radiohead를 예시로 유사 아티스트를 확인한다.

In [15]:
model = ItemBasedCF().fit(train)
print("행렬 크기(사용자, 아티스트):", model.matrix.shape)
print("train 희소성:", f"{1 - model.matrix.nnz / np.prod(model.matrix.shape):.2%}")
print("유사도 행렬 메모리:", round(model.similarity.nbytes / 1024**2, 1), "MiB")
print(model.similar_artists("radiohead", k=K).to_string(index=False))


행렬 크기(사용자, 아티스트): (973, 4824)
train 희소성: 95.21%
유사도 행렬 메모리: 88.8 MiB
          artist_name  similarity
                 beck    0.624136
             the cure    0.620282
               pixies    0.616823
                  air    0.613469
    the white stripes    0.612429
               r.e.m.    0.607951
             coldplay    0.605125
           portishead    0.601017
the smashing pumpkins    0.598042
       massive attack    0.594984


## 4.2 사용자별 아티스트 추천
추천 점수는 다음과 같이 계산한다.

score(u, i) = Σ_j [preference(u, j) × similarity(j, i)]

j는 사용자가 학습 데이터에서 들은 아티스트다. 선호도가 높은 아티스트와 비슷하거나 여러 청취 아티스트와 연관된 후보일수록 점수가 높아진다. 점수는 유사도 합으로 나누지 않은 가중합이며, 예측 별점이나 확률을 의미하지 않는다.

추천 후보는 학습 데이터에 존재하는 전체 아티스트로 설정한다. 사용자가 학습 데이터에서 들은 아티스트는 제외하고, 평가용으로 숨긴 아티스트는 후보로 남긴다. 양의 점수를 가진 후보만 추천하며, 10명을 채우지 못하더라도 인기순으로 보충하지 않는다.

첫 번째 사용자를 예시로 기존 청취 이력, 추천 결과, 1위 추천에 기여한 청취 아티스트를 확인한다.

In [16]:
USER_ID = model.users[0]
print("사용자:", USER_ID)
print("\ntrain에서 많이 들은 아티스트")
print(train[train["user_id"] == USER_ID].nlargest(10, "plays")[["artist_name", "plays"]].to_string(index=False))
example = model.recommend(USER_ID, k=K)
print("\n추천 결과")
print(example.to_string(index=False))
if not example.empty:
    target = example.iloc[0]["artist_name"]
    print("\n1위 추천에 기여한 기존 청취 아티스트:", target)
    print(model.explain(USER_ID, target).to_string(index=False))


사용자: user_000001

train에서 많이 들은 아티스트
       artist_name  plays
        underworld    779
             björk    448
             plaid    444
          röyksopp    268
         radiohead    247
      flying lotus    226
         loco dice    188
  boards of canada    180
london elektricity    179
     the black dog    149

추천 결과
    user_id  rank           artist_name     score
user_000001     1                   air 70.437782
user_000001     2 the chemical brothers 69.610474
user_000001     3  thievery corporation 67.835892
user_000001     4             goldfrapp 67.628181
user_000001     5        massive attack 67.589485
user_000001     6     nightmares on wax 67.563850
user_000001     7                moloko 67.169388
user_000001     8         groove armada 67.124901
user_000001     9             dj shadow 67.037781
user_000001    10            portishead 64.446487

1위 추천에 기여한 기존 청취 아티스트: air
 listened_artist  preference  similarity  contribution
        röyksopp    5.594711    0.609

## 4.3 전체 사용자 추천 및 저장
전체 사용자의 추천 결과를 생성하고 중복 여부, 기존 청취 아티스트 제외 여부, 후보 범위를 확인한다.

공통 평가 형식인 user_id, rank, artist_name을 recs_memory_cf.csv로 저장한다. 추천 점수를 포함한 결과는 recs_memory_cf_scores.csv로 별도 저장한다.


In [17]:
scored_recs = model.recommend_all(k=K)
recs_cf = scored_recs[["user_id", "rank", "artist_name"]].copy()
assert not recs_cf.duplicated(["user_id", "artist_name"]).any()
assert recs_cf.merge(train, on=["user_id", "artist_name"]).empty
assert recs_cf.groupby("user_id").size().le(K).all()
assert set(recs_cf["artist_name"]) <= set(model.artists)
recs_cf.to_csv(OUTPUT_DIR / "recs_memory_cf.csv", index=False)
scored_recs.to_csv(OUTPUT_DIR / "recs_memory_cf_scores.csv", index=False)
counts = recs_cf.groupby("user_id").size().reindex(model.users, fill_value=0)
print("추천 행 수:", len(recs_cf))
print("10명 미만 추천 사용자 수:", int(counts.lt(K).sum()))


추천 행 수: 9730
10명 미만 추천 사용자 수: 0


## 4.4 인기순 추천과 기본 성능 비교
03_split.ipynb와 동일하게 학습 데이터의 청취 사용자 수를 기준으로 인기순 추천을 생성한다. 각 사용자에 대해 기존 청취 아티스트를 제외하고 10명을 추천한다.

인기순 추천과 기본 협업 필터링을 동일한 평가 데이터에서 비교하고, 결과를 metrics_memory_cf.csv로 저장한다. 이 단계의 test 지표는 기본 성능 확인에 사용하며, 추가 실험의 설정 선택에는 사용하지 않는다.


In [18]:
popular = train.groupby("artist_name")["user_id"].count().sort_values(ascending=False).index
rows = []
for user_id, group in train.groupby("user_id"):
    listened = set(group["artist_name"])
    candidates = [artist for artist in popular if artist not in listened][:K]
    rows.extend((user_id, rank, artist) for rank, artist in enumerate(candidates, start=1))
recs_popular = pd.DataFrame(rows, columns=["user_id", "rank", "artist_name"])

metrics = []
for name, recs in [("Popular", recs_popular), ("Item-based CF", recs_cf)]:
    print("\n" + name)
    precision, recall = evaluate(recs, test, k=K)
    metrics.append({"model": name, f"Precision@{K}": precision, f"Recall@{K}": recall})
comparison = pd.DataFrame(metrics)
comparison.to_csv(OUTPUT_DIR / "metrics_memory_cf.csv", index=False)
print("\n" + comparison.to_string(index=False))



Popular
Precision@10: 0.2953
Recall@10: 0.0546

Item-based CF
Precision@10: 0.4643
Recall@10: 0.1002

        model  Precision@10  Recall@10
      Popular      0.295272   0.054632
Item-based CF      0.464337   0.100189


## 5. 추가 실험: 유사 아티스트 이웃 수에 따른 비교
기존의 전체 이웃 방식에 더해 상위 10·20·50명 방식의 검증 성능을 비교한다. 선호도는 기존 log(1 + plays), 유사도는 코사인으로 유지하며 최종 추천 수는 모두 10명이다.

설정 선택에는 기존 학습 데이터 내부에서 분리한 검증 데이터를 사용한다. 검증에서 선택한 설정으로 전체 학습 데이터에서 최종 추천을 생성한다. 추가 실험의 설정 선택과 추천 생성에는 test를 사용하지 않는다.

### 5.1. 실험용 학습·검증 데이터 분리
기존 train을 사용자별로 학습용 80%, 검증용 20%로 나누고 난수값을 42로 고정한다. 검증에만 존재하는 아티스트도 정답에 남겨, 학습에서 관측하지 못한 아티스트를 추천하지 못하는 한계를 평가에 반영한다.

In [19]:
fit_parts, valid_parts = [], []
for user_id, group in train.groupby("user_id"):
    fit_group, valid_group = train_test_split(
        group, test_size=0.2, random_state=42
    )
    fit_parts.append(fit_group)
    valid_parts.append(valid_group)

fit_data = pd.concat(fit_parts, ignore_index=True)
valid_data = pd.concat(valid_parts, ignore_index=True)
assert fit_data.merge(valid_data, on=["user_id", "artist_name"]).empty
print("전체 학습 데이터:", len(train))
print("실험용 학습 데이터:", len(fit_data))
print("검증 데이터:", len(valid_data))
print("검증에만 있는 아티스트:", len(set(valid_data["artist_name"]) - set(fit_data["artist_name"])))


전체 학습 데이터: 224925
실험용 학습 데이터: 179568
검증 데이터: 45357
검증에만 있는 아티스트: 3


## 5.2. 후보별 이웃 수 제한
각 추천 후보 아티스트에 대해 유사도가 높은 이웃만 남긴다. 사용자가 해당 이웃을 들었다면 로그 선호도와 유사도의 곱을 후보의 점수에 반영한다. 후보별 이웃 수와 사용자에게 추천할 아티스트 수는 구분한다.

all은 전체 이웃을 사용한다. top10, top20, top50은 양의 유사도를 가진 이웃을 각각 최대 10·20·50명 사용한다. 동점 이웃은 아티스트 이름순으로 선택한다.

후보별 이웃 선택 후 유사도 행렬은 비대칭일 수 있다. 기존 history @ similarity 계산에서 열이 추천 후보를 나타내도록 행렬을 전치한다. 원래 모델의 유사도 행렬은 변경하지 않는다.

In [20]:
def limit_neighbors(base_model, n_neighbors=None):
    variant = copy.copy(base_model)
    if n_neighbors is None:
        return variant

    similarity = base_model.similarity
    limited = np.zeros_like(similarity)
    for candidate_idx in range(len(base_model.artists)):
        scores = similarity[candidate_idx]
        order = np.argsort(-scores, kind="stable")
        neighbors = order[scores[order] > 0][:n_neighbors]
        limited[candidate_idx, neighbors] = scores[neighbors]

    variant.similarity = limited.T.copy()
    return variant


## 5.3. 검증 성능 비교 및 설정 선택
실험용 학습 데이터만으로 유사도를 계산하고 네 설정을 동일한 검증 데이터에서 평가한다. Precision@10이 높은 설정을 선택하며, 동점이면 Recall@10을 사용한다. 두 지표가 모두 같으면 설정 순서상 먼저 나온 방식을 선택한다.

In [21]:
validation_base = ItemBasedCF().fit(fit_data)
rows = []
for method, n_neighbors in SETTINGS.items():
    variant = limit_neighbors(validation_base, n_neighbors)
    recs = variant.recommend_all(k=K)
    print(f"\n===== {method} =====")
    precision, recall = evaluate(
        recs[["user_id", "rank", "artist_name"]], valid_data, k=K
    )
    rows.append({
        "method": method,
        "Precision@10": precision,
        "Recall@10": recall,
    })

del variant
validation_results = pd.DataFrame(rows)
ranked = validation_results.sort_values(
    ["Precision@10", "Recall@10"],
    ascending=[False, False], kind="stable"
)
best_method = ranked.iloc[0]["method"]
validation_results["selected"] = validation_results["method"].eq(best_method)
print("\n검증 결과")
print(validation_results.to_string(index=False))
print("\n선택된 설정:", best_method)



===== all =====
Precision@10: 0.2647
Recall@10: 0.0735

===== top10 =====
Precision@10: 0.2505
Recall@10: 0.0704

===== top20 =====
Precision@10: 0.2655
Recall@10: 0.0752

===== top50 =====
Precision@10: 0.2847
Recall@10: 0.0819

검증 결과
method  Precision@10  Recall@10  selected
   all      0.264748   0.073494     False
 top10      0.250462   0.070422     False
 top20      0.265468   0.075198     False
 top50      0.284687   0.081866      True

선택된 설정: top50


## 5.4. 최종 추천 생성 및 저장

검증에서 선택한 설정으로 전체 train을 사용해 최종 추천 결과를 생성한다. 중복 여부, 기존 청취 아티스트 제외 여부, 후보 범위와 추천 순위를 확인한다.

공통 평가 형식인 `user_id, rank, artist_name`으로 최종 추천 결과를 `recs_memory_cf_selected.csv`에 저장한다. 설정별 검증 지표와 선택 여부는 `validation_memory_cf.csv`로 별도 저장한다.

In [22]:
final_base = ItemBasedCF().fit(train)
final_model = limit_neighbors(final_base, SETTINGS[best_method])
scored_recs = final_model.recommend_all(k=K)
final_recs = scored_recs[["user_id", "rank", "artist_name"]].copy()

assert final_recs.merge(train, on=["user_id", "artist_name"]).empty
assert not final_recs.duplicated(["user_id", "artist_name"]).any()
assert set(final_recs["artist_name"]) <= set(train["artist_name"])
counts = final_recs.groupby("user_id").size().reindex(final_base.users, fill_value=0)
assert counts.le(K).all()
assert final_recs.groupby("user_id")["rank"].apply(
    lambda ranks: ranks.tolist() == list(range(1, len(ranks) + 1))
).all()

OUTPUT_DIR.mkdir(exist_ok=True)
validation_results.to_csv(OUTPUT_DIR / "validation_memory_cf.csv", index=False)
final_recs.to_csv(OUTPUT_DIR / "recs_memory_cf_selected.csv", index=False)

print("최종 설정:", best_method)
print("추천 사용자 수:", final_recs["user_id"].nunique())
print("추천 행 수:", len(final_recs))
print("10명 미만 추천 사용자 수:", int(counts.lt(K).sum()))
print("\n추천 예시")
print(final_recs.head(10).to_string(index=False))
print("\n저장 완료")
print(OUTPUT_DIR / "validation_memory_cf.csv")
print(OUTPUT_DIR / "recs_memory_cf_selected.csv")


최종 설정: top50
추천 사용자 수: 973
추천 행 수: 9730
10명 미만 추천 사용자 수: 0

추천 예시
    user_id  rank                artist_name
user_000001     1                  dj shadow
user_000001     2                   dj krush
user_000001     3       thievery corporation
user_000001     4                   four tet
user_000001     5                     tricky
user_000001     6 the future sound of london
user_000001     7             massive attack
user_000001     8       kruder & dorfmeister
user_000001     9          nightmares on wax
user_000001    10                      unkle

저장 완료
outputs\validation_memory_cf.csv
outputs\recs_memory_cf_selected.csv
